# Actividad 7. Validación de resultados

Contrasta las cifras publicadas en la aplicación con el dataset procesado y
con las fuentes. Cada comprobación se guarda en
`resultados/tablas/validacion_resultados.csv`.

La regla es simple: **una cifra solo se publica si se puede recalcular**.

## Paso 1. Carga de datos

In [1]:
import json
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATOS = RAIZ / "app" / "datos"
TABLAS = RAIZ / "resultados" / "tablas"


def cargar(nombre):
    return json.loads((DATOS / nombre).read_text(encoding="utf-8"))


df = pd.read_csv(RAIZ / "datos" / "procesados" / "EMDAT_integrado_WDI_validado.csv", low_memory=False)
filtros = cargar("filtros.json")
metadatos = cargar("metadatos.json")
geojson = cargar("paises.geojson")
socio = cargar("paises_socio.json")["paises"]
eventos = cargar("eventos_pais.json")
owid = cargar("owid.json")["paises"]
mexico = cargar("mexico_estados.json")
incendios = cargar("incendios_mexico.json")

print(f"Dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas")

Dataset: 10,907 filas × 69 columnas


## Paso 2. Registro de comprobaciones

In [2]:
comprobaciones = []


def revisar(nombre, obtenido, esperado, tolerancia=0):
    if isinstance(obtenido, float) or isinstance(esperado, float):
        ok = abs(float(obtenido) - float(esperado)) <= tolerancia
    else:
        ok = obtenido == esperado
    comprobaciones.append(
        {"comprobacion": nombre, "obtenido": obtenido, "esperado": esperado,
         "resultado": "OK" if ok else "REVISAR"}
    )
    return ok

## Paso 3. Integridad del dataset

In [3]:
revisar("Filas del dataset", len(df), 10907)
revisar("Columnas del dataset", df.shape[1], 69)
revisar("Duplicados de DisNo.", int(df["DisNo."].duplicated().sum()), 0)
revisar("Eventos en metadatos", metadatos["n_eventos"], 10907)
revisar("Territorios en metadatos", metadatos["n_territorios"], 175)

True

## Paso 4. Totales mundiales

In [4]:
ev_app = sum(f[2] or 0 for filas in filtros["datos"].values() for f in filas)
fal_app = sum(f[3] or 0 for filas in filtros["datos"].values() for f in filas)
afe_app = sum(f[4] or 0 for filas in filtros["datos"].values() for f in filas)
dan_app = sum(f[5] or 0 for filas in filtros["datos"].values() for f in filas)

fal_ds = float(df["Total Deaths"].sum())
afe_ds = float(df["Total Affected"].sum())
dan_ds = float(df["Total Damage, Adjusted ('000 MXN)"].sum() * 1000)

revisar("Eventos: filtros vs dataset", ev_app, len(df))
revisar("Fallecidos: filtros vs dataset", fal_app, fal_ds)
revisar("Afectados: filtros vs dataset", afe_app, afe_ds)
revisar("Daño: filtros vs dataset", dan_app, dan_ds, 1)

print(f"Fallecidos: {fal_ds:,.0f}   Afectados: {afe_ds:,.0f}   Daño: {dan_ds / 1e12:,.2f} billones MXN")

Fallecidos: 1,737,592   Afectados: 4,903,433,684   Daño: 108.20 billones MXN


## Paso 5. Cobertura geográfica

In [5]:
codigos = {f["properties"].get("codigo") for f in geojson["features"]}
en_mapa = int(df["ISO_union"].isin(codigos).sum())
fuera = len(df) - en_mapa

revisar("Polígonos en el GeoJSON", len(geojson["features"]), 175)
revisar("Eventos con polígono", en_mapa, 10588)
revisar("Eventos sin polígono", fuera, 319)

print(f"En el mapa: {en_mapa:,} ({100 * en_mapa / len(df):.2f} %)   Fuera: {fuera:,} ({100 * fuera / len(df):.2f} %)")

En el mapa: 10,588 (97.08 %)   Fuera: 319 (2.92 %)


## Paso 6. Cobertura socioeconómica

In [6]:
territorios_emdat = set(filtros["datos"])
con_pib = sum(1 for v in socio.values() if v.get("pib_mxn"))
con_ambos = sum(
    1
    for iso in territorios_emdat
    if socio.get(iso, {}).get("pib_mxn") and socio.get(iso, {}).get("poblacion")
)

revisar("Territorios con PIB en el WDI", con_pib, 264)
revisar("Territorios de EM-DAT con PIB y población", con_ambos, 201)
revisar("PIB de México (billones)", round(socio["MEX"]["pib_mxn"] / 1e12, 2), 38.07)

True

## Paso 7. Validación cruzada de México

In [7]:
mex_emdat = eventos["paises"]["MEX"]
fal_mex = sum(f[2] or 0 for f in mex_emdat)
afe_mex = sum(f[3] or 0 for f in mex_emdat)
dan_mex = sum(f[4] or 0 for f in mex_emdat)
afe_owid = sum(f[1] for f in owid["MEX"])
diferencia = abs(afe_mex - afe_owid) / ((afe_mex + afe_owid) / 2) * 100

revisar("México: eventos EM-DAT", len(mex_emdat), 171)
revisar("México: fallecidos EM-DAT", int(fal_mex), 2631)
revisar("México: afectados EM-DAT", int(afe_mex), 15538601)
revisar("México: daño EM-DAT (billones)", round(dan_mex / 1e12, 2), 1.60)
revisar("México: diferencia con OWID (%)", round(diferencia, 2), 0.15, 0.01)

True

## Paso 8. CENAPRED (México por estados)

In [8]:
total = mexico["total"]

revisar("Estados con datos", len(mexico["estados"]), 32)
revisar("Filas estado-evento", total["eventos"], 4195)
revisar("Defunciones CENAPRED", int(total["defunciones"]), 5181)
revisar("Daño CENAPRED (mil M MXN)", round(total["dano_mxn"] / 1e9, 1), 687.0)
revisar("Viviendas dañadas", int(total["viviendas"]), 2576168)
revisar("Fichas destacadas", len(mexico["destacados"]), 4)

True

## Paso 9. CONAFOR

In [9]:
puntos = cargar("incendios_puntos.geojson")

revisar("Incendios CONAFOR", incendios["total"]["incendios"], 74853)
revisar("Hectáreas CONAFOR (millones)", round(incendios["total"]["hectareas"] / 1e6, 2), 7.29)
revisar("Incendios de 50 ha o más", len(puntos["features"]), 14052)

True

## Paso 10. Rankings mundiales

In [10]:
def puesto(valores, objetivo):
    return 1 + sum(v > objetivo for v in valores)


filas = []
for iso, registros in filtros["datos"].items():
    total_iso = sum(f[2] or 0 for f in registros)
    if total_iso == 0:
        continue
    filas.append(
        {
            "iso": iso,
            "eventos": total_iso,
            "fallecidos": sum(f[3] or 0 for f in registros),
            "afectados": sum(f[4] or 0 for f in registros),
            "dano": sum(f[5] or 0 for f in registros),
        }
    )

mexico_fila = next(f for f in filas if f["iso"] == "MEX")
for campo, esperado in [("eventos", 11), ("fallecidos", 63), ("afectados", 35), ("dano", 10)]:
    actual = puesto([f[campo] for f in filas], mexico_fila[campo])
    revisar(f"Ranking de México en {campo}", actual, esperado)

print(f"Territorios con eventos: {len(filas)}")

Territorios con eventos: 220


## Paso 11. Reporte

In [11]:
reporte = pd.DataFrame(comprobaciones)
TABLAS.mkdir(parents=True, exist_ok=True)
reporte.to_csv(TABLAS / "validacion_resultados.csv", index=False)

print(reporte.to_string(index=False))
print()
print(f"Comprobaciones: {len(reporte)}")
print(f"Correctas     : {(reporte['resultado'] == 'OK').sum()}")
print(f"Por revisar   : {(reporte['resultado'] != 'OK').sum()}")

                             comprobacion     obtenido     esperado resultado
                        Filas del dataset 1.090700e+04 1.090700e+04        OK
                     Columnas del dataset 6.900000e+01 6.900000e+01        OK
                     Duplicados de DisNo. 0.000000e+00 0.000000e+00        OK
                     Eventos en metadatos 1.090700e+04 1.090700e+04        OK
                 Territorios en metadatos 1.750000e+02 1.750000e+02        OK
              Eventos: filtros vs dataset 1.090700e+04 1.090700e+04        OK
           Fallecidos: filtros vs dataset 1.737592e+06 1.737592e+06        OK
            Afectados: filtros vs dataset 4.903434e+09 4.903434e+09        OK
                 Daño: filtros vs dataset 1.081999e+14 1.081999e+14        OK
                  Polígonos en el GeoJSON 1.750000e+02 1.750000e+02        OK
                     Eventos con polígono 1.058800e+04 1.058800e+04        OK
                     Eventos sin polígono 3.190000e+02 3.190000e

## Conclusión

Si todas las comprobaciones quedan en `OK`, las cifras publicadas en la
aplicación coinciden con el dataset y con las fuentes. Cualquier `REVISAR`
señala una cifra que hay que corregir antes de exponerla.